# Cuisine Classifier — Training Notebook

Trains a cuisine classifier on the **What's Cooking** Kaggle dataset (39,774 recipes, 20 cuisines) and benchmarks four model families:

1. `MultinomialNB`              — baseline
2. `LogisticRegression`         — strong linear baseline
3. `LinearSVC` + `CalibratedClassifierCV` — calibrated linear SVM
4. `MLPClassifier`              — neural network

Outputs an artifact compatible with the FastAPI wrapper at `app/models/ml_models.py::CuisineClassifierModel`:

```python
{"model": <fitted_classifier>, "vectorizer": <fitted_tfidf>}
```

## How to run on Kaggle

1. Create a new Kaggle Notebook.
2. **Add data** → search `whats-cooking` (the Kaggle competition dataset) → Add.
3. Run all cells.
4. The trained `.pkl` lands in `/kaggle/working/cuisine_model.pkl` — download from the **Output** tab.


## 1. Imports & config

In [ ]:
import os
import json
import time
import pickle
import zipfile
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Imports ready.")

## 2. Load the dataset

The `whats-cooking` competition data ships as `train.json` (labeled) and `test.json` (unlabeled, for the competition leaderboard). We only need `train.json` — we'll make our own held-out split from it.

This loader handles both extracted JSON and zipped JSON, and tries a couple of plausible Kaggle paths.

In [ ]:
CANDIDATE_DIRS = [
    "/kaggle/input/whats-cooking",
    "/kaggle/input/whats-cooking-dataset",
]

train_path = None
for d in CANDIDATE_DIRS:
    if not os.path.isdir(d):
        continue
    for fname in ["train.json", "train.json.zip"]:
        fp = os.path.join(d, fname)
        if os.path.exists(fp):
            train_path = fp
            break
    if train_path:
        break

if train_path is None:
    raise FileNotFoundError(
        "Could not find train.json. On Kaggle: Add Data → search 'whats-cooking' → Add."
    )

print(f"Loading: {train_path}")
if train_path.endswith(".zip"):
    with zipfile.ZipFile(train_path) as z:
        with z.open("train.json") as f:
            raw = json.load(f)
else:
    with open(train_path) as f:
        raw = json.load(f)

df = pd.DataFrame(raw)
print(f"Loaded {len(df):,} recipes.")
df.head()

## 3. Exploratory data analysis

In [ ]:
print(f"Total recipes      : {len(df):,}")
print(f"Unique cuisines    : {df['cuisine'].nunique()}")
print(f"Unique ingredients : {len(set(i for row in df['ingredients'] for i in row)):,}")
df['n_ingredients'] = df['ingredients'].apply(len)
df['n_ingredients'].describe().round(2)

In [ ]:
# Cuisine class distribution
counts = df['cuisine'].value_counts()
fig, ax = plt.subplots(figsize=(13, 5))
counts.plot(kind='bar', ax=ax, color=sns.color_palette("viridis", len(counts)))
ax.set_title("Recipes per cuisine", fontsize=14, fontweight='bold')
ax.set_xlabel("Cuisine")
ax.set_ylabel("# recipes")
for i, v in enumerate(counts.values):
    ax.text(i, v + 80, str(v), ha='center', fontsize=8)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f"\nMax/min class ratio: {counts.max() / counts.min():.1f}x — significant imbalance.")

In [ ]:
# Ingredient distribution + top ingredients
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

df['n_ingredients'].hist(bins=40, ax=axes[0], color='teal', edgecolor='white')
axes[0].set_title("Ingredients per recipe", fontweight='bold')
axes[0].set_xlabel("# ingredients")
axes[0].set_ylabel("# recipes")
axes[0].axvline(df['n_ingredients'].median(), color='red', linestyle='--',
                label=f"median={int(df['n_ingredients'].median())}")
axes[0].legend()

ing_counter = Counter(i for row in df['ingredients'] for i in row)
top20 = pd.Series(dict(ing_counter.most_common(20)))
top20.sort_values().plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title("Top 20 ingredients (corpus-wide)", fontweight='bold')
axes[1].set_xlabel("# occurrences")

plt.tight_layout()
plt.show()

## 4. Preprocessing

- Pretty-print labels (`southern_us` → `Southern Us`, `cajun_creole` → `Cajun Creole`).
- Build a single text column per recipe by joining ingredients with spaces — exactly how the API will call us at inference time.

In [ ]:
def pretty_label(c: str) -> str:
    return " ".join(p.capitalize() for p in c.split("_"))

df['cuisine_label'] = df['cuisine'].apply(pretty_label)
df['text'] = df['ingredients'].apply(lambda items: " ".join(s.lower().strip() for s in items))

CLASSES = sorted(df['cuisine_label'].unique())
print(f"{len(CLASSES)} classes:")
print(CLASSES)
df[['cuisine_label', 'text']].head(3)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'].values,
    df['cuisine_label'].values,
    test_size=0.2,
    stratify=df['cuisine_label'].values,
    random_state=RANDOM_STATE,
)
print(f"Train: {len(X_train):,}   Test: {len(X_test):,}")

## 5. TF-IDF vectorization

Bigrams matter — they catch multi-word ingredients (`fried rice`, `tikka masala`, `spring roll`).

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    sublinear_tf=True,
    strip_accents='unicode',
)

t0 = time.time()
Xtr = vectorizer.fit_transform(X_train)
Xte = vectorizer.transform(X_test)
print(f"TF-IDF fit+transform: {time.time() - t0:.1f}s")
print(f"Train matrix shape  : {Xtr.shape}")
print(f"Vocab size          : {len(vectorizer.vocabulary_):,}")
print(f"Sparsity            : {1 - Xtr.nnz / (Xtr.shape[0] * Xtr.shape[1]):.4%}")

## 6. Train & evaluate four models

Each model is timed for both training and prediction so we can weigh accuracy against latency.

In [ ]:
results = {}

def train_and_eval(name: str, clf):
    print(f"\n=== {name} ===")
    t0 = time.time()
    clf.fit(Xtr, y_train)
    train_t = time.time() - t0

    t0 = time.time()
    preds = clf.predict(Xte)
    pred_t = time.time() - t0

    acc = accuracy_score(y_test, preds)
    macro = f1_score(y_test, preds, average='macro')
    weighted = f1_score(y_test, preds, average='weighted')

    results[name] = {
        "model": clf,
        "preds": preds,
        "accuracy": acc,
        "macro_f1": macro,
        "weighted_f1": weighted,
        "train_time": train_t,
        "pred_time": pred_t,
    }
    print(f"accuracy   : {acc:.4f}")
    print(f"macro F1   : {macro:.4f}")
    print(f"weighted F1: {weighted:.4f}")
    print(f"train time : {train_t:.1f}s")
    print(f"predict t. : {pred_t:.2f}s")
    return clf

In [ ]:
# Model 1: Multinomial Naive Bayes (baseline)
train_and_eval("MultinomialNB", MultinomialNB())

In [ ]:
# Model 2: Logistic Regression
train_and_eval(
    "LogisticRegression",
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        n_jobs=-1,
        solver='lbfgs',
    ),
)

In [ ]:
# Model 3: Linear SVC + sigmoid calibration (so we get predict_proba)
train_and_eval(
    "LinearSVC+Calibrated",
    CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", max_iter=2000),
        cv=3,
        method='sigmoid',
    ),
)

In [ ]:
# Model 4: MLPClassifier (the neural network)
train_and_eval(
    "MLPClassifier",
    MLPClassifier(
        hidden_layer_sizes=(256, 128),
        activation='relu',
        solver='adam',
        max_iter=25,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=RANDOM_STATE,
        verbose=False,
    ),
)

## 7. Side-by-side comparison

In [ ]:
comp_df = pd.DataFrame({
    name: {
        "accuracy"    : r["accuracy"],
        "macro_f1"    : r["macro_f1"],
        "weighted_f1" : r["weighted_f1"],
        "train_s"     : r["train_time"],
        "predict_s"   : r["pred_time"],
    }
    for name, r in results.items()
}).T.sort_values("macro_f1", ascending=False)

display(comp_df.style.format({
    "accuracy": "{:.4f}",
    "macro_f1": "{:.4f}",
    "weighted_f1": "{:.4f}",
    "train_s": "{:.1f}",
    "predict_s": "{:.2f}",
}).background_gradient(subset=["accuracy", "macro_f1", "weighted_f1"], cmap='Greens'))

In [ ]:
# Bar chart of all metrics for all models
metrics_df = comp_df[["accuracy", "macro_f1", "weighted_f1"]]
fig, ax = plt.subplots(figsize=(11, 5))
metrics_df.plot(kind='bar', ax=ax, width=0.8,
                color=['#3498db', '#e74c3c', '#2ecc71'])
ax.set_title("Model comparison — accuracy / macro-F1 / weighted-F1",
             fontsize=13, fontweight='bold')
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.legend(loc='lower right')
plt.xticks(rotation=15, ha='right')
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8, padding=2)
plt.tight_layout()
plt.show()

In [ ]:
# Train + predict time tradeoff
fig, ax = plt.subplots(figsize=(9, 5))
for name, r in results.items():
    ax.scatter(r["train_time"], r["macro_f1"], s=180, label=name, alpha=0.85, edgecolor='black')
    ax.annotate(name, (r["train_time"], r["macro_f1"]),
                xytext=(7, 7), textcoords='offset points', fontsize=9)
ax.set_xlabel("Training time (s)")
ax.set_ylabel("Macro F1")
ax.set_title("Cost vs. quality", fontsize=13, fontweight='bold')
ax.set_xscale('log')
plt.tight_layout()
plt.show()

## 8. Confusion matrices (normalized per true class)

In [ ]:
labels_sorted = sorted(np.unique(y_test))
n_models = len(results)
ncols = 2
nrows = (n_models + 1) // 2
fig, axes = plt.subplots(nrows, ncols, figsize=(20, 9 * nrows))
axes = axes.flatten() if n_models > 1 else [axes]

for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r["preds"], labels=labels_sorted, normalize='true')
    sns.heatmap(
        cm, ax=ax, cmap="Blues",
        xticklabels=labels_sorted, yticklabels=labels_sorted,
        cbar=True, annot=True, fmt=".2f", annot_kws={"size": 7},
        vmin=0, vmax=1,
    )
    ax.set_title(f"{name} — normalized confusion matrix\n(macro-F1 = {r['macro_f1']:.3f})",
                 fontweight='bold')
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)

for ax in axes[len(results):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

## 9. Best model — detailed per-class breakdown

In [ ]:
best_name = comp_df.index[0]
best = results[best_name]
print(f"Best model: {best_name}")
print(f"Macro F1  : {best['macro_f1']:.4f}\n")
print(classification_report(y_test, best["preds"], digits=3))

In [ ]:
# Per-class F1 for the best model
per_class_f1 = pd.Series(
    {
        label: f1_score(y_test == label, best["preds"] == label)
        for label in labels_sorted
    }
).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#e74c3c' if v < 0.6 else '#f39c12' if v < 0.75 else '#27ae60'
          for v in per_class_f1.values]
per_class_f1.plot(kind='barh', ax=ax, color=colors, edgecolor='black')
ax.axvline(per_class_f1.mean(), color='black', linestyle='--', alpha=0.5,
           label=f"mean = {per_class_f1.mean():.3f}")
ax.set_title(f"{best_name} — per-class F1", fontsize=13, fontweight='bold')
ax.set_xlabel("F1 score")
ax.set_xlim(0, 1)
ax.legend()
for i, v in enumerate(per_class_f1.values):
    ax.text(v + 0.005, i, f"{v:.3f}", va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 10. Top discriminating ingredients per cuisine (LogReg)

In [ ]:
if "LogisticRegression" in results:
    lr = results["LogisticRegression"]["model"]
    feature_names = np.array(vectorizer.get_feature_names_out())
    classes = lr.classes_
    n_top = 12

    n_classes = len(classes)
    ncols = 4
    nrows = (n_classes + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(20, 4 * nrows))
    axes = axes.flatten()

    for ax, cls, coef in zip(axes, classes, lr.coef_):
        top_idx = np.argsort(coef)[-n_top:]
        ax.barh(feature_names[top_idx], coef[top_idx], color='steelblue', edgecolor='black')
        ax.set_title(cls, fontsize=11, fontweight='bold')
        ax.tick_params(axis='y', labelsize=8)

    for ax in axes[n_classes:]:
        ax.axis('off')

    fig.suptitle("Top 12 ingredient features per cuisine (Logistic Regression coefficients)",
                 fontsize=14, fontweight='bold', y=1.005)
    plt.tight_layout()
    plt.show()
else:
    print("LogisticRegression not in results — skipping feature inspection.")

## 11. Hand-crafted dish-name sanity test

The training data is *ingredient-level* but the API receives *dish-name-level* menus. This sanity set checks that the model still does the right thing on dish-name inputs.

In [ ]:
sanity_set = [
    (["paneer tikka", "naan", "biryani"],                "Indian"),
    (["dal", "samosa", "tandoori chicken"],              "Indian"),
    (["fried rice", "spring roll", "manchurian"],        "Chinese"),
    (["dumplings", "noodles", "wonton soup"],            "Chinese"),
    (["pizza margherita", "lasagna", "tiramisu"],        "Italian"),
    (["spaghetti carbonara", "bruschetta", "risotto"],   "Italian"),
    (["tacos", "guacamole", "burrito"],                  "Mexican"),
    (["enchilada", "quesadilla", "salsa"],               "Mexican"),
    (["sushi", "miso soup", "tempura"],                  "Japanese"),
    (["pad thai", "tom yum", "green curry"],             "Thai"),
    (["kimchi", "bulgogi", "bibimbap"],                  "Korean"),
    (["croissant", "ratatouille", "coq au vin"],         "French"),
    (["fish and chips", "shepherds pie", "scones"],      "British"),
    (["moussaka", "tzatziki", "gyro"],                   "Greek"),
    (["pho", "banh mi", "spring rolls"],                 "Vietnamese"),
    (["paella", "tortilla", "gazpacho"],                 "Spanish"),
]

texts  = [" ".join(items) for items, _ in sanity_set]
truths = [label for _, label in sanity_set]
sanity_X = vectorizer.transform(texts)
preds = best["model"].predict(sanity_X)

# predict_proba is available on all four model families we trained
probas = best["model"].predict_proba(sanity_X)
classes = best["model"].classes_
top_conf = probas.max(axis=1)

results_df = pd.DataFrame({
    "menu": [", ".join(items) for items, _ in sanity_set],
    "expected": truths,
    "predicted": preds,
    "confidence": top_conf.round(3),
    "ok": [t == p for t, p in zip(truths, preds)],
})

acc = results_df["ok"].mean()
print(f"Sanity accuracy: {results_df['ok'].sum()}/{len(results_df)} = {acc:.1%}\n")
display(results_df.style.apply(
    lambda r: ['background-color: #d4edda' if r['ok'] else 'background-color: #f8d7da'] * len(r),
    axis=1,
))

## 12. Save the artifact

Saved in the format the FastAPI wrapper at `app/models/ml_models.py::CuisineClassifierModel` expects:

```python
{"model": <fitted_classifier>, "vectorizer": <fitted_tfidf>}
```

Extra metadata keys are ignored by the wrapper but useful for traceability.

In [ ]:
out_path = "/kaggle/working/cuisine_model.pkl"

artifact = {
    "model": best["model"],
    "vectorizer": vectorizer,
    # metadata (wrapper ignores unknown keys)
    "best_model_name": best_name,
    "classes": list(best["model"].classes_),
    "metrics": {
        "accuracy"    : float(best["accuracy"]),
        "macro_f1"    : float(best["macro_f1"]),
        "weighted_f1" : float(best["weighted_f1"]),
    },
    "training_dataset": "kaggle whats-cooking",
    "n_train_samples": int(len(X_train)),
}

with open(out_path, "wb") as f:
    pickle.dump(artifact, f)

size_mb = os.path.getsize(out_path) / 1024 / 1024
print(f"Saved: {out_path}")
print(f"Size : {size_mb:.2f} MB")
print(f"Best model: {best_name}  (macro-F1 = {best['macro_f1']:.4f})")

In [ ]:
# Reload + smoke test (proves the artifact deserializes correctly)
with open(out_path, "rb") as f:
    reloaded = pickle.load(f)

clf = reloaded["model"]
vec = reloaded["vectorizer"]

example = ["paneer tikka", "naan", "biryani"]
X = vec.transform([" ".join(example)])
pred = clf.predict(X)[0]
prob = float(clf.predict_proba(X)[0].max())

print(f"Input    : {example}")
print(f"Predicted: {pred}  (confidence={prob:.3f})")
print(f"\nArtifact metadata:")
for k, v in reloaded.items():
    if k not in ("model", "vectorizer"):
        print(f"  {k}: {v}")

## 13. Integration into the FastAPI app

Once `cuisine_model.pkl` is downloaded from the **Output** tab on Kaggle:

1. **Place the file** at `app/ml/models/cuisine_model.pkl`.

2. **Implement the wrapper TODO** at `app/models/ml_models.py:442`:

   ```python
   text = " ".join(menu_items)
   X = self.vectorizer.transform([text])
   pred = self.model.predict(X)[0]
   proba = self.model.predict_proba(X)[0]
   return {
       "cuisine": str(pred),
       "confidence": round(float(proba.max()), 3),
       "model": "ai",
   }
   ```

3. **Expand the `CuisineType` enum** in `app/models/schemas.py` to include all 20 trained classes — the model's predictions must match enum values:

   ```python
   class CuisineType(str, Enum):
       BRAZILIAN     = "Brazilian"
       BRITISH       = "British"
       CAJUN_CREOLE  = "Cajun Creole"
       CHINESE       = "Chinese"
       FILIPINO      = "Filipino"
       FRENCH        = "French"
       GREEK         = "Greek"
       INDIAN        = "Indian"
       IRISH         = "Irish"
       ITALIAN       = "Italian"
       JAMAICAN      = "Jamaican"
       JAPANESE      = "Japanese"
       KOREAN        = "Korean"
       MEXICAN       = "Mexican"
       MOROCCAN      = "Moroccan"
       RUSSIAN       = "Russian"
       SOUTHERN_US   = "Southern Us"
       SPANISH       = "Spanish"
       THAI          = "Thai"
       VIETNAMESE    = "Vietnamese"
   ```

4. **Set `USE_AI_MODELS=True`** in `.env`, restart the server.

5. **Test**:
   ```bash
   curl -X POST http://localhost:8000/restaurant/cuisine-classifier \
        -H "Content-Type: application/json" \
        -d '{"menu_items": ["paneer tikka", "naan", "biryani"]}'
   ```

   Expected: `{"cuisine_type": "Indian", "model_used": "ai", ...}`.

6. **Frontend** — update flag emoji map in `frontend/index.js` to include the new cuisines.
